# 분석 목표

- 여러 제조사별 무선청소기의 가격과 성능에 대한 데이터를 수집하고 분석하여 각각의 상황에 맞는 제품군 파악

In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

In [2]:
# Windows용 한글 폰트 오류 해결
from matplotlib import font_manager, rc
from pathlib import Path
font_path = "C:/Windows/Fonts/malgun.ttf"
if not Path(font_path).is_file():
    font_path = next(
        (path for path in (
            "/System/Library/Fonts/Supplemental/AppleGothic.ttf",
            "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
            "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
        ) if Path(path).is_file()),
        font_path,
    )
font_name = font_manager.FontProperties(fname = font_path).get_name()
rc("font", family = font_name)

# 데이터 수집

In [4]:
url = "https://prod.danawa.com/list/?cate=102207&shortcutKeyword=%EB%AC%B4%EC%84%A0%EC%B2%AD%EC%86%8C%EA%B8%B0"

driver = webdriver.Chrome()
driver.get(url)
driver.implicitly_wait(3)
soup = BeautifulSoup(driver.page_source, "lxml")

In [5]:
prod_items = soup.select("div.main_prodlist li.prod_item")

In [18]:
len(prod_items)

30

In [9]:
# 상품명
prod_items[0].select_one("p.prod_name > a").string.strip()

'아이닉 i50 아이타워'

In [12]:
# 스펙 목록
prod_items[0].select_one("div.spec-box--full > div.spec_list").text.strip()

'핸디스틱청소기/무선/흡입형/흡입력: 520W/흡입력: 45,000Pa/스테이션 기능: 먼지비움, 충전, 액서서리수납, 스탠드거치/브러쉬: 전방향바닥/침구/솔형/틈새/먼지봉투: 1.5L/[배터리] 사용시간: 1시간05분(최대)/분리형(1개)/[청소] LED라이트/UV살균/BLDC모터/헤파필터/무게: 2.7kg/크기(가로x세로x깊이): 250x1040x260mm'

In [17]:
# 가격
prod_items[5].select_one("p.price_sect strong").string.replace(",", "").strip()

'1170000'

In [21]:
prod_data = {"상품명" : [],
             "스펙목록" : [],
             "가격" : []}

prod_items = soup.select("div.main_prodlist li.prod_item")

for item in prod_items:
    # 상품명
    title = item.select_one("p.prod_name > a").string.strip()
    # 스펙 목록
    spec_list = item.select_one("div.spec-box--full > div.spec_list").text.strip()
    # 가격
    price = item.select_one("p.price_sect strong").string.replace(",", "").strip()

    prod_data["상품명"].append(title)
    prod_data["스펙목록"].append(spec_list)
    prod_data["가격"].append(price)

In [22]:
len(prod_data["상품명"]), len(prod_data["스펙목록"]), len(prod_data["가격"])

(30, 30, 30)

In [23]:
prod_data["상품명"][:5]

['아이닉 i50 아이타워',
 'LG전자 오브제컬렉션 코드제로 A9S AU927CWD',
 'LG전자 오브제컬렉션 코드제로 A9S AX958BWE',
 'LG전자 코드제로 A9 Air AS9000WR',
 'LG전자 오브제컬렉션 코드제로 A9S AX9604WE']

In [24]:
prod_data["스펙목록"][:5]

['핸디스틱청소기/무선/흡입형/흡입력: 520W/흡입력: 45,000Pa/스테이션 기능: 먼지비움, 충전, 액서서리수납, 스탠드거치/브러쉬: 전방향바닥/침구/솔형/틈새/먼지봉투: 1.5L/[배터리] 사용시간: 1시간05분(최대)/분리형(1개)/[청소] LED라이트/UV살균/BLDC모터/헤파필터/무게: 2.7kg/크기(가로x세로x깊이): 250x1040x260mm',
 '핸디스틱청소기/무선/흡입+물걸레(동시)/흡입력: 260W/스테이션 기능: 먼지비움, 충전, UVC LED, 액서서리수납, 스탠드거치/먼지비움시간: 30초/브러쉬: 바닥/물걸레: 일반/솔형/틈새/먼지봉투: 2.5L/[배터리] 사용시간: 1시간(최대)/분리형(1개)/리튬이온/[청소] 싸이클론흡입/자동물공급/스마트인버터모터/워셔블헤파필터/무게: 2.5kg/크기(가로x세로x깊이): 250x1120x260mm',
 '핸디스틱청소기/무선/흡입+물걸레(동시)/흡입력: 280W/스테이션 기능: 먼지비움, 충전, 액서서리수납, 스탠드거치/먼지비움시간: 30초/브러쉬: 와이드바닥/물걸레: 스팀, 고온, 일반/침구/솔형/틈새/먼지봉투: 2.5L/[배터리] 사용시간: 30분(최대)/분리형(2개)/리튬이온/[청소] LED라이트/싸이클론흡입/오토스탑앤고/자동물공급/스마트인버터모터/워셔블헤파필터/무게: 2.49kg/크기(가로x세로x깊이): 300x1100x245mm',
 '핸디스틱청소기/무선/흡입형/흡입력: 150W/스테이션 기능: 충전, 스탠드거치, 액서서리&배터리거치/브러쉬: 바닥/솔형/틈새/틈새(내장형)/[배터리] 사용시간: 40분(최대)/분리형(1개)/리튬이온/[청소] 싸이클론흡입/스마트인버터모터/무게: 1.97kg/크기(가로x세로x깊이): 251x1026x242mm',
 '핸디스틱청소기/무선/흡입형/흡입력: 250W/스테이션 기능: 먼지비움, 충전, UVC LED, 액서서리수납, 스탠드거치/먼지비움시간: 30초/브러쉬: 와이드바닥/물걸레: 별매/솔형/틈새/먼지봉투: 2.5L/[배터리] 사용시간: 30분(최대)/분

In [25]:
prod_data["가격"][:5]

['297220', '702490', '978740', '362907', '683420']

In [26]:
# 10페이지까지 반복
prod_data = {"상품명" : [],
             "스펙목록" : [],
             "가격" : []}

driver = webdriver.Chrome()
driver.get(url)

for i in range(10):
    time.sleep(2)
    soup = BeautifulSoup(driver.page_source, "lxml")
    
    prod_items = soup.select("div.main_prodlist li.prod_item")
    
    for item in prod_items:
        # 상품명
        title = item.select_one("p.prod_name > a").string.strip()
        # 스펙 목록
        spec_list = item.select_one("div.spec-box--full > div.spec_list").text.strip()
        # 가격
        price = item.select_one("p.price_sect strong").string.replace(",", "").strip()
    
        prod_data["상품명"].append(title)
        prod_data["스펙목록"].append(spec_list)
        prod_data["가격"].append(price)

    if i != 9:
        driver.find_element(By.CSS_SELECTOR, "a.now_on + a").click()

# 데이터 전처리

## 데이터 확인

In [27]:
df = pd.DataFrame(prod_data)

In [28]:
df.head()

,상품명,스펙목록,가격
0,아이닉 i50 아이타워,"핸디스틱청소기/무선/흡입형/흡입력: 520W/흡입력: 45,000Pa/스테이션 기능...",297210
1,LG전자 오브제컬렉션 코드제로 A9S AU927CWD,"핸디스틱청소기/무선/흡입+물걸레(동시)/흡입력: 260W/스테이션 기능: 먼지비움,...",702490
2,LG전자 오브제컬렉션 코드제로 A9S AX958BWE,"핸디스틱청소기/무선/흡입+물걸레(동시)/흡입력: 280W/스테이션 기능: 먼지비움,...",978730
3,LG전자 코드제로 A9 Air AS9000WR,"핸디스틱청소기/무선/흡입형/흡입력: 150W/스테이션 기능: 충전, 스탠드거치, 액...",362907
4,LG전자 오브제컬렉션 코드제로 A9S AX9604WE,"핸디스틱청소기/무선/흡입형/흡입력: 250W/스테이션 기능: 먼지비움, 충전, UV...",683420


In [29]:
df.shape

(300, 3)

In [30]:
df.to_csv("./data/danawa.csv", index = False)

In [31]:
df.dtypes

상품명     object
스펙목록    object
가격      object
dtype: object

In [32]:
# 스펙목록 전처리
def clean_text(text):
    text = text.replace("\n", "")
    text = text.replace("\t", "")
    text = text.replace(" ", "")
    return text

In [33]:
df["스펙목록"] = df["스펙목록"].map(clean_text)

In [34]:
df.head()

,상품명,스펙목록,가격
0,아이닉 i50 아이타워,"핸디스틱청소기/무선/흡입형/흡입력:520W/흡입력:45,000Pa/스테이션기능:먼지...",297210
1,LG전자 오브제컬렉션 코드제로 A9S AU927CWD,"핸디스틱청소기/무선/흡입+물걸레(동시)/흡입력:260W/스테이션기능:먼지비움,충전,...",702490
2,LG전자 오브제컬렉션 코드제로 A9S AX958BWE,"핸디스틱청소기/무선/흡입+물걸레(동시)/흡입력:280W/스테이션기능:먼지비움,충전,...",978730
3,LG전자 코드제로 A9 Air AS9000WR,"핸디스틱청소기/무선/흡입형/흡입력:150W/스테이션기능:충전,스탠드거치,액서서리&배...",362907
4,LG전자 오브제컬렉션 코드제로 A9S AX9604WE,"핸디스틱청소기/무선/흡입형/흡입력:250W/스테이션기능:먼지비움,충전,UVCLED,...",683420


In [36]:
df["가격"].unique()

array(['297210', '702490', '978730', '362907', '683420', '1170000',
       '154900', '641630', '639840', '196000', '1290820', '462800',
       '696000', '190000', '423860', '649790', '735000', '772470',
       '198000', '299000', '361830', '729000', '550610', '1058040',
       '820800', '1099000', '744920', '219100', '596060', '87980',
       '443710', '364900', '498960', '453890', '412220', '459690',
       '792723', '328860', '805000', '205270', '486130', '565890',
       '827690', '1060930', '177650', '743870', '179000', '106910',
       '449100', '341100', '29240', '426000', '69000', '106000', '464520',
       '1109780', '205880', '295720', '800990', '832970', '548000',
       '39590', '24240', '394200', '423200', '40000', '145000', '42780',
       '303450', '148720', '88100', '1009890', '158000', '1276940',
       '99000', '921300', '687300', '820010', '663890', '118000',
       '892000', '148000', '159000', '1104770', '349000', '564000',
       '37900', '186230', '580310', '50220

In [39]:
df = df[df["가격"] != "가격비교예정"]

In [40]:
# 가격 컬럼을 정수형으로 변환
df["가격"] = df["가격"].astype(int)

In [41]:
df.dtypes

상품명     object
스펙목록    object
가격       int32
dtype: object

## 회사명, 모델명 정리

- 상품명 데이터는 회사명 + 모델명의 형식임
    - 브랜드와 모델명으로 분리

In [46]:
df.iloc[0, 0].split(" ", 1) # 첫 번째 공백에 대해서만 구분하도록

['아이닉', 'i50 아이타워']

In [50]:
titles = df["상품명"].str.split(n = 1)

In [54]:
df["회사명"] = titles.str[0]
df["제품"] = titles.str[1]

In [55]:
df.head()

,상품명,스펙목록,가격,회사명,제품
0,아이닉 i50 아이타워,"핸디스틱청소기/무선/흡입형/흡입력:520W/흡입력:45,000Pa/스테이션기능:먼지...",297210,아이닉,i50 아이타워
1,LG전자 오브제컬렉션 코드제로 A9S AU927CWD,"핸디스틱청소기/무선/흡입+물걸레(동시)/흡입력:260W/스테이션기능:먼지비움,충전,...",702490,LG전자,오브제컬렉션 코드제로 A9S AU927CWD
2,LG전자 오브제컬렉션 코드제로 A9S AX958BWE,"핸디스틱청소기/무선/흡입+물걸레(동시)/흡입력:280W/스테이션기능:먼지비움,충전,...",978730,LG전자,오브제컬렉션 코드제로 A9S AX958BWE
3,LG전자 코드제로 A9 Air AS9000WR,"핸디스틱청소기/무선/흡입형/흡입력:150W/스테이션기능:충전,스탠드거치,액서서리&배...",362907,LG전자,코드제로 A9 Air AS9000WR
4,LG전자 오브제컬렉션 코드제로 A9S AX9604WE,"핸디스틱청소기/무선/흡입형/흡입력:250W/스테이션기능:먼지비움,충전,UVCLED,...",683420,LG전자,오브제컬렉션 코드제로 A9S AX9604WE


In [56]:
df = df.drop("상품명", axis = 1)

In [57]:
df.head()

,스펙목록,가격,회사명,제품
0,"핸디스틱청소기/무선/흡입형/흡입력:520W/흡입력:45,000Pa/스테이션기능:먼지...",297210,아이닉,i50 아이타워
1,"핸디스틱청소기/무선/흡입+물걸레(동시)/흡입력:260W/스테이션기능:먼지비움,충전,...",702490,LG전자,오브제컬렉션 코드제로 A9S AU927CWD
2,"핸디스틱청소기/무선/흡입+물걸레(동시)/흡입력:280W/스테이션기능:먼지비움,충전,...",978730,LG전자,오브제컬렉션 코드제로 A9S AX958BWE
3,"핸디스틱청소기/무선/흡입형/흡입력:150W/스테이션기능:충전,스탠드거치,액서서리&배...",362907,LG전자,코드제로 A9 Air AS9000WR
4,"핸디스틱청소기/무선/흡입형/흡입력:250W/스테이션기능:먼지비움,충전,UVCLED,...",683420,LG전자,오브제컬렉션 코드제로 A9S AX9604WE


## 스펙 목록 살펴보기

In [58]:
df.iloc[0, 0]

'핸디스틱청소기/무선/흡입형/흡입력:520W/흡입력:45,000Pa/스테이션기능:먼지비움,충전,액서서리수납,스탠드거치/브러쉬:전방향바닥/침구/솔형/틈새/먼지봉투:1.5L/[배터리]사용시간:1시간05분(최대)/분리형(1개)/[청소]LED라이트/UV살균/BLDC모터/헤파필터/무게:2.7kg/크기(가로x세로x깊이):250x1040x260mm'

In [66]:
df.iloc[1, 0].split("/")

['핸디스틱청소기',
 '무선',
 '흡입+물걸레(동시)',
 '흡입력:260W',
 '스테이션기능:먼지비움,충전,UVCLED,액서서리수납,스탠드거치',
 '먼지비움시간:30초',
 '브러쉬:바닥',
 '물걸레:일반',
 '솔형',
 '틈새',
 '먼지봉투:2.5L',
 '[배터리]사용시간:1시간(최대)',
 '분리형(1개)',
 '리튬이온',
 '[청소]싸이클론흡입',
 '자동물공급',
 '스마트인버터모터',
 '워셔블헤파필터',
 '무게:2.5kg',
 '크기(가로x세로x깊이):250x1120x260mm']

- 카테고리 : 스펙 리스트의 첫 번째 항목에 위치

- 사용시간 : "사용시간: 00시간" 과 같이 사용시간 이라는 문구가 명시되어 있음

- 흡입력 : "흡입력:000W" 와 같이 흡입력 이라는 문구가 명시되어 있음

In [67]:
# test
spec_list = df.iloc[0, 0].split("/")

In [68]:
category = spec_list[0]
category

'핸디스틱청소기'

In [69]:
for spec in spec_list:
    if "사용시간" in spec:
        use_time_spec = spec
        use_time_spec = use_time_spec.split(":")[1]

    elif "흡입력" in spec:
        suction_spec = spec
        suction_spec = suction_spec.split(":")[1]

print(use_time_spec)
print(suction_spec)

1시간05분(최대)
45,000Pa


In [73]:
# 반복문
category_list = []
use_time_list = []
suction_list = []

for spec_data in df["스펙목록"]:
    # / 기준으로 분리
    spec_list = spec_data.split("/")

    # 카테고리 추출
    category = spec_list[0]
    category_list.append(category)

    # 사용시간, 흡입력 추출
    use_time_spec = None
    suction_spec = None

    for spec in spec_list:
        if "사용시간" in spec:
            use_time_spec = spec.split(":")[1]

        elif "흡입력" in spec:
            suction_spec = spec.split(":")[1]

    use_time_list.append(use_time_spec)
    suction_list.append(suction_spec)

In [75]:
category_list[:5]

['핸디스틱청소기', '핸디스틱청소기', '핸디스틱청소기', '핸디스틱청소기', '핸디스틱청소기']

In [76]:
use_time_list[:5]

['1시간05분(최대)', '1시간(최대)', '30분(최대)', '40분(최대)', '30분(최대)']

In [77]:
suction_list[:5]

['45,000Pa', '260W', '280W', '150W', '250W']